# 05 워드클라우드 — 보조 그림(부록)

그룹별 2종 — 기본 빈도 / 공통어 제거(4그룹 고빈도 교집합). 본증거는 03 막대그래프, 이건 부록
- `_` 결합 토큰은 집계 유지, 시각화 라벨만 공백으로


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !pip install wordcloud


In [ ]:
from pathlib import Path
import os, re, unicodedata
from collections import Counter
import pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from wordcloud import WordCloud

try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists(): raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
os.chdir(PROJECT_DIR); RESULT_DIR = PROJECT_DIR / 'outputs'
GROUPS = ['경제','통신·보도','정치색','지상파']
def nn(p): return unicodedata.normalize('NFC', p.name)
# 워드클라우드용 한글 폰트 경로(파일 경로 필요)
FONT_PATH = None
for cand in ['NanumGothic','Malgun','AppleGothic','NanumBarunGothic']:
    hit = [f.fname for f in fm.fontManager.ttflist if cand.lower() in f.name.lower()]
    if hit: FONT_PATH = hit[0]; break
print('font_path:', FONT_PATH)
if FONT_PATH is None:
    raise FileNotFoundError('한글 폰트 없음 — Colab: !apt-get -qq install fonts-nanum 후 런타임 재시작. 폰트 없이 그리면 네모로 깨짐')


In [ ]:
# --- 분석코퍼스(in_universe) 토큰 로드 ---
cc = sorted(p for p in RESULT_DIR.iterdir() if re.match(r'^분석코퍼스_언론사_\d{6}_\d{6}\.csv$', nn(p)))
if not cc: raise FileNotFoundError('01 분석코퍼스 먼저')
CORP = cc[-1]; PERIOD = re.search(r'(\d{6}_\d{6})', nn(CORP)).group(1)
df = pd.read_csv(CORP, encoding='utf-8-sig', usecols=['media_group','tokens','in_universe'])
df['in_universe'] = df['in_universe'].map({'True':True,'False':False,True:True,False:False})
df = df[df['in_universe'] == True]
TOP_N = 100   # 고빈도 정의 — 각 그룹 상위 N

def grp_counts(g):
    c = Counter()
    for s in df[df['media_group']==g]['tokens']: c.update(str(s).split())
    return c
counts = {g: grp_counts(g) for g in GROUPS}
# 공통어 = 4그룹 각 상위 100 교집합
top100 = {g: {w for w,_ in counts[g].most_common(TOP_N)} for g in GROUPS}
common = set.intersection(*top100.values())
print('4그룹 공통 고빈도어(제거 대상):', sorted(common))


In [ ]:
# --- 그룹별 2종(기본 / 공통어 제거) 워드클라우드 ---
def draw(ax, freq, title):
    show = {w.replace('_',' '): n for w,n in freq.items() if n > 0}   # 집계는 _ 유지, 라벨만 공백
    if not show:
        ax.axis('off'); ax.set_title(title + ' (표시 토큰 없음)'); return
    wc = WordCloud(font_path=FONT_PATH, width=600, height=400, background_color='white',
                   max_words=120).generate_from_frequencies(show)
    ax.imshow(wc); ax.axis('off'); ax.set_title(title)

fig, axes = plt.subplots(len(GROUPS), 2, figsize=(11, 4*len(GROUPS)))
for i, g in enumerate(GROUPS):
    base = dict(counts[g])
    nocommon = {w:n for w,n in base.items() if w not in common}
    draw(axes[i,0], base, f'{g} — 기본 빈도')
    draw(axes[i,1], nocommon, f'{g} — 공통어 제거')
fig.tight_layout(); fig.savefig(RESULT_DIR / f'워드클라우드_{PERIOD}.png', dpi=150)
print(f'저장: 워드클라우드_{PERIOD}.png (본증거는 03 막대그래프, 이건 부록)')


## 검증 체크리스트
- 그룹별 2종(기본/공통어 제거) 한글 정상 렌더
- 공통어 제거판에서 그룹 변별어가 부각되는지
- `_` 결합 토큰 집계 유지·라벨만 공백
